# Induction Evals — One-Hop
Mirrors `induction_eval.ipynb` (intensional / extensional / noise-padded intensional)
but uses the **one-hop** parameterisation from commit 8248d70:
- Query: direct successor (`Has $color1 handed the sceptre to $color2?`) — one hop
  for *both* representations.
- `n = 52 × 250 = 13 000` (wider year range, so direct-neighbour pairs are sparser
  relative to the context length).

In [ ]:
"""Generates the one-hop quiz that all models are evaluated on."""

import itertools
import string
import numpy as np
import yaml

import logging

from ordered_set import OrderedSet
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
load_dotenv("./keys.env", verbose=True)

from smolbench.induction.chromatic import (
    ChromaticIntervalsConfig,
    Prompter,
    get_random_exclusive_quiz,
    anneal_intervals,
)
from smolbench.evals import Marks
from smolbench.evals.openrouter import evaluate

template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. At the end of one's term as $role, they have a ceremony where they hand off the"
    " $role ceremonial sceptre to their successor. The following lists the people who were $role and"
    " the years they were $role:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "Has $color1 handed the sceptre to $color2?"
)

extens_template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. At the end of one's term as $role, they have a ceremony where they hand off the"
    " $role ceremonial sceptre to their successor. The following lists each year and who was $role"
    " that year:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "Has $color1 handed the sceptre to $color2?"
)


def query_gen(
    labels_to_intervals: Dict[Color, Intervals],
    interval_to_label: Dict[Interval, Color],
    seed: int,
) -> Iterable[Tuple[Dict[str, str], bool]]:
    """Generates direct-succession queries.

    Yields True for each unique (predecessor, successor) pair and an equal
    number of False for randomly-sampled non-successor pairs.
    """
    rng: np.random.Generator = np.random.default_rng(seed)
    sorted_intervals = sorted(interval_to_label.items(), key=lambda item: item[0][0])
    # True cases: pairs where color2 immediately followed color1.
    true_pairs: OrderedSet = OrderedSet(
        (c1, c2)
        for ((_s1, e1), c1), ((s2, _e2), c2) in zip(sorted_intervals, sorted_intervals[1:])
    )
    for color1, color2 in true_pairs:
        yield {"color1": color1, "color2": color2}, True
    # False cases: same count, randomly sampled non-successor pairs.
    all_colors: list = list(labels_to_intervals.keys())
    false_pairs: OrderedSet = OrderedSet()
    while len(false_pairs) < len(true_pairs):
        c1, c2 = (str(c) for c in rng.choice(all_colors, size=2, replace=False))
        if (c1, c2) not in true_pairs:
            false_pairs.add((c1, c2))
    for color1, color2 in false_pairs:
        yield {"color1": color1, "color2": color2}, False


SEED: int = 1776
intens_quiz, extens_quiz, noise_intens_quiz = get_random_exclusive_quiz(
    ChromaticIntervalsConfig(
        n=int(12 * 250),
        intervals=250 // 4,
        colors=45,
        seed=SEED,
    ),
    Prompter(
        template,
        {
            "role": "Twislax",
            "parade": "Gildane",
        },
        query_gen,
        extens_template,
    ),
)

## Prompt Validation

In [ ]:
print(intens_quiz[0].prompt)

In [ ]:
print(extens_quiz[0].prompt)

In [ ]:
print(noise_intens_quiz[0].prompt)

## Decoder-Only Model
This section tests classical decoder-only models.

In [ ]:
decode_intens_eval: Marks = evaluate(
    intens_quiz, "mistralai/devstral-small", SEED
)

In [ ]:
decode_extens_eval: Marks = evaluate(
    extens_quiz, "mistralai/devstral-small", SEED
)

In [ ]:
decode_noise_intens_eval: Marks = evaluate(
    noise_intens_quiz, "mistralai/devstral-small", SEED
)

In [ ]:
# Prints out results.
print(
    decode_intens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid
)
print(
    decode_extens_eval.correct, decode_extens_eval.incorrect, decode_extens_eval.invalid
)
print(
    decode_noise_intens_eval.correct, decode_noise_intens_eval.incorrect, decode_noise_intens_eval.invalid
)
# Serializes results
with open('results/one_hop_decode_intens.yaml', 'w') as file:
    yaml.dump(
        decode_intens_eval, file, default_flow_style=False, indent=4
    )
with open('results/one_hop_decode_extens.yaml', 'w') as file:
    yaml.dump(
        decode_extens_eval, file, default_flow_style=False, indent=4
    )
with open('results/one_hop_decode_noise_intens.yaml', 'w') as file:
    yaml.dump(
        decode_noise_intens_eval, file, default_flow_style=False, indent=4
    )

## CoT Model
This section tests a CoT model.

In [ ]:
cot_intens_eval: Marks = evaluate(
    intens_quiz, "google/gemma-3-27b-it", SEED, extra_args={"reasoning": {"enabled": True}}
)

In [ ]:
cot_extens_eval: Marks = evaluate(
    extens_quiz, "google/gemma-3-27b-it", SEED, extra_args={"reasoning": {"enabled": True}}
)

In [ ]:
cot_noise_intens_eval: Marks = evaluate(
    noise_intens_quiz, "google/gemma-3-27b-it", SEED, extra_args={"reasoning": {"enabled": True}}
)

In [ ]:
print(cot_intens_eval.correct, cot_intens_eval.incorrect, cot_intens_eval.invalid)
print(cot_extens_eval.correct, cot_extens_eval.incorrect, cot_extens_eval.invalid)
print(cot_noise_intens_eval.correct, cot_noise_intens_eval.incorrect, cot_noise_intens_eval.invalid)
# Serializes results
with open('results/one_hop_cot_intens.yaml', 'w') as file:
    yaml.dump(
        cot_intens_eval, file, default_flow_style=False, indent=4
    )
with open('results/one_hop_cot_extens.yaml', 'w') as file:
    yaml.dump(
        cot_extens_eval, file, default_flow_style=False, indent=4
    )
with open('results/one_hop_cot_noise_intens.yaml', 'w') as file:
    yaml.dump(
        cot_noise_intens_eval, file, default_flow_style=False, indent=4
    )

## MoE Model
This section tests an MoE model.

In [ ]:
moe_intens_eval: Marks = evaluate(intens_quiz, "qwen/qwen3-30b-a3b-instruct-2507", SEED)

In [ ]:
moe_extens_eval: Marks = evaluate(extens_quiz, "qwen/qwen3-30b-a3b-instruct-2507", SEED)

In [ ]:
moe_noise_intens_eval: Marks = evaluate(noise_intens_quiz, "qwen/qwen3-30b-a3b-instruct-2507", SEED)

In [ ]:
print(moe_intens_eval.correct, moe_intens_eval.incorrect, moe_intens_eval.invalid)
print(moe_extens_eval.correct, moe_extens_eval.incorrect, moe_extens_eval.invalid)
print(moe_noise_intens_eval.correct, moe_noise_intens_eval.incorrect, moe_noise_intens_eval.invalid)
# Serializes results
with open('results/one_hop_moe_intens.yaml', 'w') as file:
    yaml.dump(
        moe_intens_eval, file, default_flow_style=False, indent=4
    )
with open('results/one_hop_moe_extens.yaml', 'w') as file:
    yaml.dump(
        moe_extens_eval, file, default_flow_style=False, indent=4
    )
with open('results/one_hop_moe_noise_intens.yaml', 'w') as file:
    yaml.dump(
        moe_noise_intens_eval, file, default_flow_style=False, indent=4
    )